# Project 4 — Trợ lý hỏi–đáp RAG trên văn bản pháp quy FMCG
## Phần 3 — Sinh câu trả lời có trích dẫn & Đánh giá bám nguồn

**Bối cảnh.** Khâu cuối của pipeline RAG: ghép các chunk truy hồi được (Phần 2) với một LLM để sinh câu trả lời **kèm trích dẫn nguồn**, và **tự từ chối** khi tài liệu không chứa câu trả lời.

**Đầu vào.** `data/chunks.json` (291 chunk) + `data/embeddings.npy` (ma trận 291×768 đã chuẩn hóa L2, từ Phần 2).

**Nội dung notebook này.**
1. Nạp dữ liệu, embedding và cấu hình LLM (Gemini API — gọi kèm retry/backoff cho lỗi 503).
2. Truy hồi ngữ nghĩa (tái sử dụng hàm `search` của Phần 2).
3. **Dựng ngữ cảnh & prompt:** ghép top-k chunk kèm metadata trích dẫn; prompt ràng buộc chỉ trả lời trong ngữ cảnh, trả lời từng phần, ưu tiên bản sửa đổi, và fallback khi không tìm thấy.
4. Chạy bộ 6 câu hỏi (5 trong phạm vi + 1 ngoài phạm vi) và lưu artifact `eval_runs.json` (ghim input–output cùng một lần chạy).
5. **Đánh giá bám nguồn (faithfulness):** đối chiếu từng trích dẫn với chunk gốc trên hai trục — *groundedness* và *độ chính xác trích dẫn*.

> Chi tiết lý thuyết: `Tong_hop_kien_thuc_4.md` (mục I, IV-bis).

## 1. Nạp dữ liệu, embedding và cấu hình LLM

Khóa API đọc từ biến môi trường (`.env`, đã loại trừ trong `.gitignore`) — không hard-code, không commit.

In [1]:
import json
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv

In [2]:
load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

# Load chunks và embeddings
print("Đang tải dữ liệu...")
with open('data/chunks.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

embeddings = np.load('data/embeddings.npy')

# Load mô hình nhúng E5
print("Đang tải mô hình E5...")
e5_model = SentenceTransformer('intfloat/multilingual-e5-base')

Đang tải dữ liệu...
Đang tải mô hình E5...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 2. Truy hồi ngữ nghĩa

Tái sử dụng hàm `search` của Phần 2: mã hóa câu hỏi (tiền tố `"query: "`), tính tích vô hướng với ma trận embedding, lấy top-k.

In [3]:
def search(query, top_k=5):
    q = e5_model.encode(["query: " + query], normalize_embeddings=True)
    scores = embeddings @ q[0]
    idx = np.argsort(-scores)[:top_k]
    return [(chunks[i], float(scores[i])) for i in idx]

## 3. Dựng ngữ cảnh & prompt

`build_context` ghép mỗi chunk kèm khối `[Luật]/[Điều]/[Nội dung]`, và **chỉ** thêm dòng ghi chú sửa đổi khi chunk thực sự có (`sua_doi`/`sua_doi_cho` khác rỗng) — tránh đưa trường trống gây nhiễu. `build_prompt` áp bộ quy tắc: ràng buộc kiến thức, trả lời từng phần, ưu tiên bản sửa đổi, fallback.

In [4]:
def build_context(chunks_topk_with_scores):
    context_str = "<context>\n"
    
    # Unpack tuple thành chunk và score
    for i, (chunk, score) in enumerate(chunks_topk_with_scores):
        context_str += f"--- Chunk {i+1} ---\n"
        context_str += f"[Luật]: {chunk.get('bo_luat', '')} ({chunk.get('so_hieu', '')}).\n"
        context_str += f"[Điều]: {chunk.get('so_dieu', '')}.\n"
        context_str += f"[Nội dung]: {chunk.get('text_for_embedding', chunk.get('noi_dung', ''))}\n"
        
        sua_doi = chunk.get('sua_doi', '')
        if sua_doi:
            context_str += f"[Sửa đổi]: {sua_doi}\n"
            
        sua_doi_cho = chunk.get('sua_doi_cho', '')
        if sua_doi_cho:
            context_str += f"[Sửa đổi cho]: {sua_doi_cho}\n"
            
    context_str += "</context>"
    return context_str

In [5]:
def build_prompt(question, context):
    """Ghép context và question vào template prompt"""
    prompt = f"""Bạn là một trợ lý AI chuyên nghiệp hỗ trợ giải đáp thông tin pháp lý. Nhiệm vụ của bạn là trả lời câu hỏi của người dùng một cách chính xác, minh bạch và có căn cứ pháp lý vững chắc.

[QUY TẮC BẮT BUỘC]
1. RÀNG BUỘC KIẾN THỨC: Bạn CHỈ ĐƯỢC PHÉP sử dụng thông tin từ phần <context> được cung cấp dưới đây. Tuyệt đối không sử dụng kiến thức bên ngoài, không suy diễn, không tự sáng tạo thêm chi tiết.
2. TRẢ LỜI LINH HOẠT TỪNG PHẦN (PARTIAL ANSWER):
- Hãy đối chiếu câu hỏi với <context>. 
- Nếu <context> trả lời được trọn vẹn, hãy trình bày đầy đủ.
- Nếu <context> chỉ trả lời được một phần câu hỏi, hãy cung cấp thông tin cho phần có căn cứ, và BẮT BUỘC phải nói rõ phần thông tin nào chưa được tìm thấy trong tài liệu.
- XỬ LÝ FALLBACK TOÀN BỘ: Chỉ khi <context> hoàn toàn không chứa bất kỳ thông tin nào liên quan đến câu hỏi, hãy trả lời đúng một câu: "Không tìm thấy thông tin trong tài liệu." và dừng lại.
3. XỬ LÝ XUNG ĐỘT VÀ HIỆU LỰC VĂN BẢN (AMENDMENTS): 
Nếu trong <context> có chứa cả bản gốc lẫn bản sửa đổi (có ghi chú "đã được sửa đổi bởi...") của cùng một vấn đề/Điều luật, bạn BẮT BUỘC phải ưu tiên sử dụng thông tin từ bản sửa đổi (mới hơn) làm câu trả lời chính thức, đồng thời nêu rõ thông tin này đã được cập nhật/sửa đổi bởi văn bản nào.
4. QUY TẮC TRÍCH DẪN CHI TIẾT: Bất cứ thông tin nào đưa ra đều phải trích dẫn nguồn. Khi trích dẫn, ghi rõ tới Khoản/Điểm dựa vào số thứ tự có sẵn trong nội dung, và trích dẫn chi tiết nhất có thể dựa trên thẻ [Luật], [Điều].
- Định dạng trích dẫn chuẩn: "... nội dung thông tin pháp lý ... [Trích: <tên nguồn chi tiết>]".

[DỮ LIỆU CUNG CẤP]
{context}

[CÂU HỎI CỦA NGƯỜI DÙNG]
{question}

[TRẢ LỜI]"""
    return prompt

## 4. Hàm sinh câu trả lời

`call_gemini` bọc lời gọi API với **retry + exponential backoff** (nuốt lỗi 503 — server quá tải nhất thời). `answer` ghép cả pipeline và trả về dict (câu hỏi, id chunk, điểm, ngữ cảnh, câu trả lời) để ghim input–output cùng một lần chạy phục vụ đánh giá.

In [ ]:
import time
from google.genai import errors

def call_gemini(prompt, model="gemini-3.6-flash", max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = client.models.generate_content(
                model=model, contents=prompt,
                config={"temperature": 0.1})
            return resp.text
        except errors.ServerError as e:
            if attempt < max_retries - 1:
                wait = 2 ** attempt   # 1, 2, 4, 8 giây
                print(f"  (503, thử lại sau {wait}s...)")
                time.sleep(wait)
            else:
                raise

In [7]:
def answer(question, top_k=5):
    chunks_topk = search(question, top_k)
    context = build_context(chunks_topk)
    prompt = build_prompt(question, context)
    resp = call_gemini(prompt, model="gemini-3.6-flash")
    return {
        "question": question,
        "retrieved_ids": [c["chunk_id"] for c, s in chunks_topk],
        "retrieved_scores": [s for c, s in chunks_topk],
        "context": context,
        "answer": resp,
    }

## 5. Chạy bộ câu hỏi & lưu artifact

Bộ 6 câu: 5 câu trong phạm vi (định nghĩa, thủ tục, ghi nhãn, hải quan, điều đã bị sửa đổi) + 1 câu ngoài phạm vi (kiểm tra fallback). Lưu toàn bộ vào `eval_runs.json` — vì LLM sinh không tất định, phải ghim (ngữ cảnh, câu trả lời) của cùng một lần chạy.

In [8]:
questions = ["thực phẩm bao gói sẵn là gì",
           "hồ sơ tự công bố sản phẩm gồm những gì",
           "nhãn hàng hóa nhập khẩu bắt buộc ghi những nội dung nào",
           "thời hạn nộp tờ khai hải quan",
           "Nhãn hàng hóa lưu thông tại Việt Nam bắt buộc thể hiện những nội dung nào?",
           "nhân viên nghỉ thai sản mấy ngày"]

evaluate = []

for question in questions:
    evaluate.append(answer(question, top_k=5))

In [ ]:
file_path = "data/eval_runs.json"

with open(file_path, "w", encoding="utf-8") as file:
    json.dump(evaluate, file, ensure_ascii=False, indent=2)

print(f"Đã lưu thành công dữ liệu vào file: {file_path}")

Đã lưu thành công dữ liệu vào file: data/eval_runs.json


## 6. Đánh giá bám nguồn (Faithfulness)

Với mỗi trích dẫn trong câu trả lời, đối chiếu thủ công với chunk gốc trên **hai trục tách biệt**:
- **Groundedness:** nội dung có thật sự nằm trong chunk truy hồi không (chống ảo giác)?
- **Độ chính xác trích dẫn:** đường dẫn *Điều/Khoản/Điểm* có trỏ đúng chỗ chứa nội dung không?

Tách hai trục vì một câu trả lời có thể *grounded nhưng sai đường dẫn* — nội dung thật nhưng gán nhầm số điều, loại lỗi nguy hiểm nhất vì nhìn rất đáng tin.

### 6.1 Bảng đối chiếu trích dẫn

| Câu hỏi | Trích dẫn LLM đưa ra | Grounded (nội dung có trong context?) | Đường dẫn trích dẫn đúng? | Ghi chú |
|---|---|---|---|---|
| 1. thực phẩm bao gói sẵn là gì | Điều 2 Khoản 27, Luật ATTP (61/VBHN-VPQH) | ✅ | ✅ | Khớp nguyên văn khoản 27 |
| 2. hồ sơ tự công bố sản phẩm | Điều 5 Khoản 1 Điểm a, b + Khoản 3 (15/2018/NĐ-CP) | ✅ | ✅ | Mẫu 01, phiếu kiểm nghiệm ISO 17025 (part_1), tiếng Việt/công chứng (part_2) — đều khớp |
| 3. nhãn hàng nhập khẩu | NĐ 111 Điều 1 Khoản 5 Điểm a/b/c/d | ✅ | ⚠️ | Nội dung đúng (Điều 10 mới Khoản 1); đường dẫn **bỏ tầng "Điều 10 Khoản 1"** |
| 4. thời hạn nộp tờ khai hải quan | Điều 25 Khoản 1 Điểm a/b/c + Khoản 2 (54/VBHN-VPQH) | ✅ | ✅ | Đúng hết; còn tự báo thiếu Điều 69 (partial answer trung thực) |
| 5. nhãn hàng lưu thông tại VN | NĐ 111 Điều 1 Khoản 2… + NĐ 43 Điều 7 | ✅ | ⚠️ | Nội dung đúng nhưng **lệch chủ đề do retrieval trượt** + đường dẫn bỏ tầng |
| 6. nghỉ thai sản mấy ngày | (từ chối) | ✅ | — | Fallback đúng: "Không tìm thấy thông tin trong tài liệu" |

### 6.2 Chỉ số tổng hợp
- **Groundedness: 6/6** — không câu nào bịa nội dung ngoài ngữ cảnh. Fallback kích hoạt đúng cho câu ngoài phạm vi dù các chunk vẫn được cấp với điểm 0.80.
- **Độ chính xác trích dẫn:** 4/6 chính xác hoàn toàn (các câu **không** dính sửa đổi: 1, 2, 4, 6). 2/6 sai đường dẫn (câu 3, 5 — đều là chunk sửa đổi).

### 6.3 Hai phát hiện

**A. Lỗi trích dẫn ở chunk sửa đổi — do cấu trúc lồng hai tầng.**
Chunk NĐ 111 chứa đồng thời hai hệ đánh số: *(NĐ 111) Điều 1 → Khoản 5* và *(Điều 10 mới) Khoản 1 → Điểm a/b/c/d*. LLM trộn hai tầng, ghi "Điều 1 Khoản 5 Điểm a" mà bỏ mất tầng giữa "Điều 10 Khoản 1". Các câu không dính sửa đổi trích dẫn hoàn hảo → nguyên nhân là cấu trúc dữ liệu, không phải năng lực LLM.
*Hướng khắc phục:* thêm trường metadata `trich_dan_chuan` (vd "Điều 10 NĐ43, sửa bởi NĐ 111 Điều 1 Khoản 5") để LLM trích theo, không tự suy từ text lồng.

**B. Retrieval trượt ở câu 5 (nhãn hàng lưu thông tại VN).**
Nội dung đúng nằm ở `Khoản_5_part_1` (khoản 1: hàng lưu thông trong nước) nhưng top-5 lấy `part_2` (khoản 2: hàng nhập khẩu) → câu trả lời **lệch sang hàng nhập khẩu**. Đây là tác dụng phụ của việc cắt Điều dài thành nhiều part: nội dung liên quan bị phân tán, chunk đúng nhất không lọt top-k.
*Hướng khắc phục:* **parent-child retrieval** — khi một part trúng, nở về Điều cha (`parent_id` đã lưu sẵn) trước khi đưa cho LLM.